# 面试题：不用现成 RNN/LSTM/GRU 层，如何手写门控公式并比较长期依赖？

## 面试回答主线

Vanilla RNN 用 `tanh(xW_x + hW_h)` 更新单一隐藏状态，重复乘递归雅可比后容易梯度消失或爆炸。LSTM 增加 cell state，并用 input/forget/output gate 控制写入、保留与读出，使信息拥有接近线性的长期通道。GRU 把状态合并为一份，用 update/reset gate 在保留旧状态和生成候选状态之间插值，参数少于 LSTM。比较三者不能只数参数或看 shape，应在同数据、同隐藏维度下展示真实 loss、梯度沿时间的位置分布与流式分块一致性。线上还要处理 padding mask、state 生命周期、截断反传和梯度裁剪。

## 真实案例：早期客户等级决定会话升级优先级

十二段客服会话的首 token 是 `VIP` 或 `普通`，之后经过多轮相似等待与追问，最终都出现“转人工”。标签判断是否进入高优先级队列，必须保留最早等级信息。数据是脱敏教学序列，只验证长期状态机制。

In [1]:
import math  # 导入平方根以计算全局梯度范数。
import torch  # 导入 PyTorch 以手写循环单元并执行真实反向传播。
from torch import nn  # 导入基础模块与参数抽象。
import torch.nn.functional as F  # 导入稳定二分类交叉熵。
torch.set_num_threads(1)  # 小张量教学实验固定单线程以快速复现。
suffixes = [  # 构造六种真实客服过程但保持最终事件相同。
    ["提交", "售后", "等待", "客服", "回复", "继续", "等待", "再次", "追问", "转人工"],  # 售后等待路径。
    ["提交", "退款", "等待", "机器人", "回复", "仍未", "解决", "再次", "追问", "转人工"],  # 退款未解决路径。
    ["提交", "物流", "查询", "等待", "承运商", "回复", "继续", "等待", "追问", "转人工"],  # 物流查询路径。
    ["提交", "账号", "问题", "等待", "验证", "回复", "仍未", "解决", "追问", "转人工"],  # 账号问题路径。
    ["提交", "发票", "问题", "等待", "客服", "回复", "继续", "等待", "追问", "转人工"],  # 发票问题路径。
    ["提交", "优惠券", "问题", "等待", "机器人", "回复", "仍未", "解决", "追问", "转人工"],  # 优惠券问题路径。
]  # 结束六种会话后缀。
sessions = []  # 创建列表保存 VIP 与普通成对会话。
for case_index, suffix in enumerate(suffixes):  # 遍历每种相同末态的过程模板。
    sessions.append({"session_id": f"S-{case_index + 1:02d}-V", "tokens": ["VIP"] + suffix, "label": 1})  # 添加应高优先级升级的 VIP 会话。
    sessions.append({"session_id": f"S-{case_index + 1:02d}-N", "tokens": ["普通"] + suffix, "label": 0})  # 添加同过程但普通优先级的配对会话。
print("会话       长度  首token  末三token            标签")  # 输出真实案例输入表标题。
for item in sessions:  # 逐条展示决定性首 token 与相同末态。
    print(f"{item['session_id']:<10} {len(item['tokens']):>4} {item['tokens'][0]:>7} {' '.join(item['tokens'][-3:]):<16} {item['label']}")  # 输出会话摘要和监督标签。

会话       长度  首token  末三token            标签
S-01-V       11     VIP 再次 追问 转人工        1
S-01-N       11      普通 再次 追问 转人工        0
S-02-V       11     VIP 再次 追问 转人工        1
S-02-N       11      普通 再次 追问 转人工        0
S-03-V       11     VIP 等待 追问 转人工        1
S-03-N       11      普通 等待 追问 转人工        0
S-04-V       11     VIP 解决 追问 转人工        1
S-04-N       11      普通 解决 追问 转人工        0
S-05-V       11     VIP 等待 追问 转人工        1
S-05-N       11      普通 等待 追问 转人工        0
S-06-V       11     VIP 解决 追问 转人工        1
S-06-N       11      普通 解决 追问 转人工        0


## Baseline（基线）：只看最后一个事件

所有会话最后一个 token 都是“转人工”，最近事件规则无法区分客户等级。并列时固定预测高优先级，只能命中一半普通/VIP 配对样本。

In [2]:
last_event_counts = {}  # 创建最后事件到标签列表的统计映射。
for item in sessions:  # 遍历全部会话建立最近事件基线。
    last_event_counts.setdefault(item["tokens"][-1], []).append(item["label"])  # 聚合同一末事件对应的冲突标签。
baseline_predictions = []  # 保存逐会话最近事件预测。
for item in sessions:  # 按所属末事件执行多数表决。
    labels_for_event = last_event_counts[item["tokens"][-1]]  # 读取当前末事件的训练标签。
    prediction = int(sum(labels_for_event) >= len(labels_for_event) / 2)  # 并列时稳定预测高优先级。
    baseline_predictions.append(prediction)  # 保存当前会话的基线预测。
baseline_accuracy = sum(prediction == item["label"] for prediction, item in zip(baseline_predictions, sessions)) / len(sessions)  # 计算最近事件准确率。
print("末事件标签分布：", last_event_counts)  # 输出同一最后事件对应的正负冲突。
print("基线预测：", baseline_predictions)  # 输出十二条会话的实际规则结果。
print(f"只看最后事件的准确率：{baseline_accuracy:.1%}")  # 输出后续循环网络的同数据基线。

末事件标签分布： {'转人工': [1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0]}
基线预测： [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]
只看最后事件的准确率：50.0%


## 核心实现一：逐公式手写 RNN、LSTM 与 GRU cell

三个 cell 都只用 `nn.Parameter`。LSTM 一次投影四组 gate，顺序为 input、forget、candidate、output；GRU 显式计算 update、reset 和 candidate。没有调用 `nn.RNN`、`nn.LSTM` 或 `nn.GRU`。

In [3]:
class ManualRNNCell(nn.Module):  # 定义最基础的 tanh 递归单元。
    def __init__(self, input_dim, hidden_dim):  # 初始化输入和递归投影参数。
        super().__init__()  # 注册基础模块状态。
        self.input_weight = nn.Parameter(torch.randn(input_dim, hidden_dim) * 0.15)  # 创建输入到隐藏状态的权重。
        self.hidden_weight = nn.Parameter(torch.randn(hidden_dim, hidden_dim) * 0.10)  # 创建上一步隐藏状态的递归权重。
        self.bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建隐藏更新偏置。
    def forward(self, inputs, hidden):  # 按 vanilla RNN 公式更新一步状态。
        return torch.tanh(inputs @ self.input_weight + hidden @ self.hidden_weight + self.bias)  # 返回经过 tanh 的新隐藏状态。
class ManualLSTMCell(nn.Module):  # 定义带显式 cell state 的四门 LSTM 单元。
    def __init__(self, input_dim, hidden_dim):  # 初始化拼接输入到四组 gate 的参数。
        super().__init__()  # 注册基础模块状态。
        self.hidden_dim = hidden_dim  # 保存隐藏维度供拆分 gate 使用。
        self.gate_weight = nn.Parameter(torch.randn(input_dim + hidden_dim, hidden_dim * 4) * 0.10)  # 创建四门共享输入投影。
        self.gate_bias = nn.Parameter(torch.zeros(hidden_dim * 4))  # 创建四组 gate 偏置。
        with torch.no_grad():  # 初始化 forget gate 不需要进入计算图。
            self.gate_bias[hidden_dim : hidden_dim * 2] = 1.0  # 把 forget bias 设为一以鼓励训练初期保留长期状态。
    def forward(self, inputs, state):  # 按 LSTM 公式更新 hidden 和 cell。
        hidden, cell = state  # 解包上一步的读出状态与记忆状态。
        combined = torch.cat([inputs, hidden], dim=1)  # 拼接当前输入和上一步 hidden。
        gates = combined @ self.gate_weight + self.gate_bias  # 一次计算四组 gate 的预激活。
        input_gate, forget_gate, candidate, output_gate = gates.chunk(4, dim=1)  # 按隐藏维度拆分四组信号。
        input_gate = torch.sigmoid(input_gate)  # 把 input gate 限制在零到一。
        forget_gate = torch.sigmoid(forget_gate)  # 把 forget gate 限制在零到一。
        candidate = torch.tanh(candidate)  # 把候选记忆限制在负一到一。
        output_gate = torch.sigmoid(output_gate)  # 把 output gate 限制在零到一。
        new_cell = forget_gate * cell + input_gate * candidate  # 在长期通道中保留旧记忆并写入候选。
        new_hidden = output_gate * torch.tanh(new_cell)  # 用 output gate 读取新的 cell state。
        return new_hidden, new_cell  # 返回下一时刻完整 LSTM 状态。
class ManualGRUCell(nn.Module):  # 定义只有一份状态的 update/reset 门控单元。
    def __init__(self, input_dim, hidden_dim):  # 初始化 update、reset 和 candidate 参数。
        super().__init__()  # 注册基础模块状态。
        combined_dim = input_dim + hidden_dim  # 计算拼接输入与隐藏状态的维度。
        self.update_weight = nn.Parameter(torch.randn(combined_dim, hidden_dim) * 0.12)  # 创建 update gate 投影。
        self.update_bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建 update gate 偏置。
        self.reset_weight = nn.Parameter(torch.randn(combined_dim, hidden_dim) * 0.12)  # 创建 reset gate 投影。
        self.reset_bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建 reset gate 偏置。
        self.candidate_weight = nn.Parameter(torch.randn(combined_dim, hidden_dim) * 0.12)  # 创建候选状态投影。
        self.candidate_bias = nn.Parameter(torch.zeros(hidden_dim))  # 创建候选状态偏置。
    def forward(self, inputs, hidden):  # 按 GRU 公式更新一步隐藏状态。
        combined = torch.cat([inputs, hidden], dim=1)  # 拼接当前输入与旧状态供两组 gate 使用。
        update_gate = torch.sigmoid(combined @ self.update_weight + self.update_bias)  # 计算保留旧状态的 update 比例。
        reset_gate = torch.sigmoid(combined @ self.reset_weight + self.reset_bias)  # 计算生成候选时读取旧状态的 reset 比例。
        candidate_input = torch.cat([inputs, reset_gate * hidden], dim=1)  # 用 reset gate 过滤旧状态后拼接当前输入。
        candidate = torch.tanh(candidate_input @ self.candidate_weight + self.candidate_bias)  # 生成新的候选隐藏状态。
        return update_gate * hidden + (1.0 - update_gate) * candidate  # 在旧状态与新候选之间按 update gate 插值。
print("手写 cell 已创建：", ManualRNNCell.__name__, ManualLSTMCell.__name__, ManualGRUCell.__name__)  # 输出三个实际类名确认没有调用现成循环层。

手写 cell 已创建： ManualRNNCell ManualLSTMCell ManualGRUCell


## 核心实现二：序列分类器、padding mask 与真实训练

分类器逐时间步调用指定 cell。mask 为 0 时保留旧状态而不是把 PAD 写入状态；最后状态经过线性头得到优先级 logit。三个模型用相同 seed、embedding 维度、hidden 维度、轮次和 SGD 学习率训练。

In [4]:
vocabulary = sorted({token for item in sessions for token in item["tokens"]}) + ["[PAD]"]  # 收集会话 token 并添加 padding 特殊词。
token_to_id = {token: index for index, token in enumerate(vocabulary)}  # 创建稳定 token id 映射。
pad_id = token_to_id["[PAD]"]  # 保存 padding 编号供批量构造使用。
max_length = max(len(item["tokens"]) for item in sessions)  # 计算本批次最大真实序列长度。
input_ids = torch.full((len(sessions), max_length), pad_id, dtype=torch.long)  # 创建以 PAD 填充的批量 id 张量。
valid_mask = torch.zeros(len(sessions), max_length, dtype=torch.float32)  # 创建对应的有效 token mask。
for row_index, item in enumerate(sessions):  # 把每条可变长度会话写入批量张量。
    ids = [token_to_id[token] for token in item["tokens"]]  # 转换当前会话 token id。
    input_ids[row_index, : len(ids)] = torch.tensor(ids, dtype=torch.long)  # 写入真实 token 并保留尾部 PAD。
    valid_mask[row_index, : len(ids)] = 1.0  # 标记当前会话真实位置。
labels = torch.tensor([item["label"] for item in sessions], dtype=torch.float32)  # 创建高优先级监督张量。
class ManualSequenceClassifier(nn.Module):  # 定义可切换三种自研 cell 的序列分类器。
    def __init__(self, vocabulary_size, embedding_dim, hidden_dim, cell_type):  # 初始化 embedding、循环单元和分类头。
        super().__init__()  # 注册基础模块状态。
        self.hidden_dim = hidden_dim  # 保存隐藏维度用于创建初始状态。
        self.cell_type = cell_type  # 保存当前使用的递归结构名称。
        self.embedding_weight = nn.Parameter(torch.randn(vocabulary_size, embedding_dim) * 0.12)  # 创建可学习 token embedding 表。
        if cell_type == "RNN":  # 根据实验选择 vanilla RNN cell。
            self.cell = ManualRNNCell(embedding_dim, hidden_dim)  # 实例化自研 RNN 单元。
        elif cell_type == "LSTM":  # 根据实验选择四门 LSTM cell。
            self.cell = ManualLSTMCell(embedding_dim, hidden_dim)  # 实例化自研 LSTM 单元。
        else:  # 其余实验使用 GRU cell。
            self.cell = ManualGRUCell(embedding_dim, hidden_dim)  # 实例化自研 GRU 单元。
        self.classifier_weight = nn.Parameter(torch.randn(hidden_dim) * 0.12)  # 创建最终二分类权重。
        self.classifier_bias = nn.Parameter(torch.zeros(()))  # 创建标量分类偏置。
    def forward(self, ids, mask, initial_state=None, retain_inputs=False):  # 逐时间步编码并返回最终分类与状态。
        embedded = self.embedding_weight[ids]  # 把批量 token id 查表为连续输入向量。
        if retain_inputs:  # 梯度诊断需要观察每个时间步的输入梯度。
            embedded.retain_grad()  # 要求 autograd 保留非叶张量的逐位置梯度。
        batch_size = ids.shape[0]  # 读取当前批量大小。
        if self.cell_type == "LSTM":  # LSTM 需要同时初始化 hidden 和 cell state。
            hidden = torch.zeros(batch_size, self.hidden_dim) if initial_state is None else initial_state[0]  # 创建或读取初始 hidden。
            cell = torch.zeros(batch_size, self.hidden_dim) if initial_state is None else initial_state[1]  # 创建或读取初始 cell。
        else:  # RNN 与 GRU 只维护一份 hidden state。
            hidden = torch.zeros(batch_size, self.hidden_dim) if initial_state is None else initial_state  # 创建或读取初始 hidden。
            cell = None  # 用 None 明确表示当前结构没有独立 cell state。
        for time_index in range(ids.shape[1]):  # 按原序列顺序执行每个时间步。
            active = mask[:, time_index : time_index + 1]  # 取得当前时间步是否为真实 token 的列向量。
            if self.cell_type == "LSTM":  # 使用四门公式更新 LSTM 双状态。
                new_hidden, new_cell = self.cell(embedded[:, time_index, :], (hidden, cell))  # 计算当前 token 候选状态。
                hidden = active * new_hidden + (1.0 - active) * hidden  # 对 PAD 位置保留旧 hidden。
                cell = active * new_cell + (1.0 - active) * cell  # 对 PAD 位置保留旧 cell。
            else:  # RNN 与 GRU 只需更新一份隐藏状态。
                new_hidden = self.cell(embedded[:, time_index, :], hidden)  # 调用对应自研 cell 计算候选状态。
                hidden = active * new_hidden + (1.0 - active) * hidden  # 用 mask 阻止 PAD 污染最终状态。
        logits = hidden @ self.classifier_weight + self.classifier_bias  # 把最终有效状态映射为优先级 logit。
        final_state = (hidden, cell) if self.cell_type == "LSTM" else hidden  # 按结构返回流式续接所需的完整状态。
        return logits, final_state, embedded  # 返回分类分数、最终状态与可选梯度观测点。
def train_model(cell_type):  # 在完全相同设置下训练一种循环结构。
    torch.manual_seed(137)  # 固定三种模型各自的参数初始化序列。
    model = ManualSequenceClassifier(len(vocabulary), 10, 14, cell_type)  # 创建十维输入、十四维隐藏的分类器。
    trace = []  # 保存代表轮次的 BCE 与准确率。
    first_moments = {id(parameter): torch.zeros_like(parameter) for parameter in model.parameters()}  # 为每个参数创建手写 Adam 一阶动量。
    second_moments = {id(parameter): torch.zeros_like(parameter) for parameter in model.parameters()}  # 为每个参数创建手写 Adam 二阶动量。
    for step in range(1001):  # 执行真实的序列前向、反向和手写 Adam。
        logits, final_state, embedded = model(input_ids, valid_mask)  # 编码完整会话并输出优先级分数。
        loss = F.binary_cross_entropy_with_logits(logits, labels)  # 计算稳定二分类交叉熵。
        loss.backward()  # 对 embedding、循环参数和分类头执行真实反向传播。
        global_norm = math.sqrt(sum(float((parameter.grad ** 2).sum()) for parameter in model.parameters()))  # 汇总全部参数的梯度 L2 范数。
        clip_scale = min(1.0, 5.0 / (global_norm + 1e-12))  # 把过大梯度按全局范数裁到五以内。
        with torch.no_grad():  # 参数更新不进入下一轮计算图。
            for parameter in model.parameters():  # 遍历当前结构的全部参数。
                gradient = parameter.grad * clip_scale  # 应用同一个全局裁剪比例保留梯度方向。
                first_moments[id(parameter)] = 0.9 * first_moments[id(parameter)] + 0.1 * gradient  # 更新手写 Adam 一阶动量。
                second_moments[id(parameter)] = 0.999 * second_moments[id(parameter)] + 0.001 * gradient.square()  # 更新手写 Adam 二阶动量。
                corrected_first = first_moments[id(parameter)] / (1.0 - 0.9 ** (step + 1))  # 对一阶动量执行有限步偏差修正。
                corrected_second = second_moments[id(parameter)] / (1.0 - 0.999 ** (step + 1))  # 对二阶动量执行有限步偏差修正。
                parameter -= 0.02 * corrected_first / (torch.sqrt(corrected_second) + 1e-8)  # 使用手写 Adam 规则更新参数。
                parameter.grad.zero_()  # 清空梯度避免跨轮错误累积。
        if step in {0, 20, 100, 300, 700, 1000}:  # 记录可以说明收敛趋势的代表轮次。
            predictions = (torch.sigmoid(logits.detach()) >= 0.5).to(torch.float32)  # 把概率阈值化为类别。
            accuracy = float((predictions == labels).to(torch.float32).mean())  # 计算当前全批量准确率。
            trace.append((step, float(loss.detach()), accuracy))  # 保存轮次、损失和准确率。
    return model, trace  # 返回训练模型和代表轨迹。
models = {}  # 创建结构名称到训练模型的映射。
traces = {}  # 创建结构名称到训练轨迹的映射。
for cell_type in ["RNN", "LSTM", "GRU"]:  # 依次训练三个自研循环模型。
    models[cell_type], traces[cell_type] = train_model(cell_type)  # 保存当前结构的模型与轨迹。
print("轮次 | RNN loss/acc | LSTM loss/acc | GRU loss/acc")  # 输出同设置训练轨迹表标题。
for row_index in range(len(traces["RNN"])):  # 对齐三个模型的相同代表轮次。
    rnn_row = traces["RNN"][row_index]  # 读取当前 RNN 状态。
    lstm_row = traces["LSTM"][row_index]  # 读取当前 LSTM 状态。
    gru_row = traces["GRU"][row_index]  # 读取当前 GRU 状态。
    print(f"{rnn_row[0]:>4} | {rnn_row[1]:.4f}/{rnn_row[2]:.0%} | {lstm_row[1]:.4f}/{lstm_row[2]:.0%} | {gru_row[1]:.4f}/{gru_row[2]:.0%}")  # 输出损失与准确率对照。

轮次 | RNN loss/acc | LSTM loss/acc | GRU loss/acc
   0 | 0.6935/50% | 0.6933/50% | 0.6932/50%
  20 | 0.5418/100% | 0.0313/100% | 0.0583/100%
 100 | 0.0005/100% | 0.0004/100% | 0.0004/100%
 300 | 0.0001/100% | 0.0001/100% | 0.0002/100%
 700 | 0.0000/100% | 0.0000/100% | 0.0000/100%
1000 | 0.0000/100% | 0.0000/100% | 0.0000/100%


## 结果表与逐时间步梯度

除了最终准确率，对每个已训练模型把全部 logit 求和后再做一次 backward，并读取 embedding 输出在各时间步的敏感度范数。使用 logit 而不是已饱和的 BCE，避免正确率 100% 后梯度被 sigmoid 压到打印精度以下；单个小实验不能证明某结构在所有长序列上都更优。

In [5]:
final_metrics = {}  # 保存三个模型的准确率、概率和逐时间梯度。
print("结构  准确率  首token梯度  中间梯度  末token梯度  VIP/普通平均概率")  # 输出最终效果和梯度表标题。
for cell_type, model in models.items():  # 逐个诊断训练后的三种循环结构。
    model.zero_grad()  # 清空训练结束时可能残留的参数梯度。
    logits, final_state, embedded = model(input_ids, valid_mask, retain_inputs=True)  # 前向并要求保留逐位置输入梯度。
    sensitivity_objective = logits.sum()  # 用未饱和 logit 和构造输入敏感度诊断目标。
    sensitivity_objective.backward()  # 反向得到每个时间步对最终 logit 的局部敏感度。
    probabilities = torch.sigmoid(logits.detach())  # 把最终 logit 转换为高优先级概率。
    predictions = (probabilities >= 0.5).to(torch.float32)  # 使用固定阈值生成类别。
    accuracy = float((predictions == labels).to(torch.float32).mean())  # 计算最终分类准确率。
    time_gradients = embedded.grad.norm(dim=2).mean(dim=0).detach()  # 按 batch 平均每个时间步的 logit 输入敏感度范数。
    vip_average = float(probabilities[labels == 1].mean())  # 计算 VIP 会话平均高优先级概率。
    normal_average = float(probabilities[labels == 0].mean())  # 计算普通会话平均高优先级概率。
    final_metrics[cell_type] = {"accuracy": accuracy, "gradients": time_gradients, "probabilities": probabilities}  # 保存完整诊断结果。
    print(f"{cell_type:<5} {accuracy:>6.1%} {float(time_gradients[0]):>12.6f} {float(time_gradients[len(time_gradients)//2]):>10.6f} {float(time_gradients[-1]):>10.6f} {vip_average:>7.3f}/{normal_average:.3f}")  # 输出效果与三个时间位置的梯度。
print("LSTM 每个时间步梯度：", [round(float(value), 6) for value in final_metrics["LSTM"]["gradients"]])  # 输出完整梯度序列供观察长期信号。

结构  准确率  首token梯度  中间梯度  末token梯度  VIP/普通平均概率
RNN   100.0%     0.004555   0.003703   0.013206   1.000/0.000
LSTM  100.0%     0.001129   0.006353   0.170121   1.000/0.000
GRU   100.0%     0.000411   0.000992   0.017600   1.000/0.000
LSTM 每个时间步梯度： [0.001129, 0.001356, 0.002118, 0.002195, 0.004499, 0.006353, 0.011366, 0.016133, 0.029845, 0.029642, 0.170121]


## 结果解读

最近事件规则面对成对样本只能达到 50%，三种自研网络都能通过真实梯度把首 token 信息压入递归状态。LSTM 的 cell state 和 GRU 的插值门为长期保留提供了结构通道，但梯度大小仍受训练参数、序列长度和损失影响；不能把“有门”简单等价为“永不消失”。参数量、吞吐和并行能力也决定实际选择。

## 失败案例：流式分块时忘记携带 state

线上长会话常分块处理。正确做法把第一块最终 hidden（LSTM 还包括 cell）传给下一块；如果每块都重置为零，前半段的 VIP 信息消失。下面在同一训练模型上比较完整前向、正确续接和错误重置。

In [6]:
stream_model = models["LSTM"]  # 选择需要同时携带 hidden 与 cell 的训练后 LSTM。
stream_ids = input_ids[0:1]  # 选择一条 VIP 会话作为流式反例。
stream_mask = valid_mask[0:1]  # 取得对应有效位置 mask。
split_point = 5  # 在首 token 与最终转人工之间切成两个服务块。
with torch.no_grad():  # 流式一致性检查不需要梯度图。
    full_logit, full_state, _ = stream_model(stream_ids, stream_mask)  # 一次处理完整会话得到数值参考。
    first_logit, first_state, _ = stream_model(stream_ids[:, :split_point], stream_mask[:, :split_point])  # 处理第一块并取得双状态。
    carried_logit, carried_state, _ = stream_model(stream_ids[:, split_point:], stream_mask[:, split_point:], initial_state=first_state)  # 正确把 hidden 和 cell 传入第二块。
    reset_logit, reset_state, _ = stream_model(stream_ids[:, split_point:], stream_mask[:, split_point:])  # 错误地让第二块从全零状态重启。
carried_error = float((full_logit - carried_logit).abs().max())  # 量化正确分块与完整前向的差异。
reset_error = float((full_logit - reset_logit).abs().max())  # 量化状态丢失造成的分类 logit 漂移。
print(f"完整会话 logit={float(full_logit[0]):.6f}")  # 输出完整前向参考分数。
print(f"携带 hidden+cell 的第二块 logit={float(carried_logit[0]):.6f}，误差={carried_error:.9f}")  # 展示正确流式续接的数值一致性。
print(f"重置 state 的第二块 logit={float(reset_logit[0]):.6f}，误差={reset_error:.6f}")  # 展示首段长期信息丢失后的变化。

完整会话 logit=10.375673
携带 hidden+cell 的第二块 logit=10.375673，误差=0.000000000
重置 state 的第二块 logit=-11.015357，误差=21.391029


## 生产差距与落地清单

教学模型只有十二条等长序列，线上需处理 packed sequence、跨 batch state 映射、截断 BPTT、梯度裁剪、dropout 和混合精度。流式 state 必须绑定会话、模型版本和过期时间，并在会话结束后释放；不能跨用户复用。长依赖评估应按距离分桶，并与 TCN/Transformer 的吞吐、显存和任务质量共同比较。

## 最小回归测试

断言只保护公式可训练、mask、分类效果和 state 续接；训练曲线、逐时间梯度和流式对照才是主要证据。

In [7]:
assert all(traces[cell_type][-1][1] < traces[cell_type][0][1] for cell_type in traces)  # 验证三个手写循环模型的真实反向传播都降低损失。
assert all(final_metrics[cell_type]["accuracy"] > baseline_accuracy for cell_type in final_metrics)  # 验证三种结构都优于只看末事件基线。
assert all(float(final_metrics[cell_type]["gradients"][0]) > 0.0 for cell_type in final_metrics)  # 验证监督梯度实际传播到首 token。
assert token_to_id["[PAD]"] == pad_id  # 验证批量 padding 使用固定特殊 token 编号。
assert carried_error < 1e-6  # 验证携带完整 LSTM state 后分块输出与全量前向一致。
assert reset_error > 1e-3  # 固化流式分块重置状态会改变结果的失败案例。
print("最小回归测试通过：RNN/LSTM/GRU 训练、长期梯度、padding mask 与流式 state 均符合预期。")  # 输出完整顺序执行成功的明确结论。

最小回归测试通过：RNN/LSTM/GRU 训练、长期梯度、padding mask 与流式 state 均符合预期。
